# Limitless TCG - deck matchup winrates

Goal: for every pair of deck archetypes, the winrate of one against the other.

The pipeline is `tournament list -> per-tournament pairings + standings -> one row
per played game -> deck-vs-deck matrix`. Pairings say who beat whom; standings say
what each player was on; the two are joined on `(tournament, player)`.

**Running it.** Top to bottom. Every per-tournament response is cached under
`cache/`, so a second run costs almost no requests and raising `MAX_PAGES` only
pays for what is new. Delete `cache/` to force a refetch.

**Reading the result.** `matchup_df` has one row per *ordered* pair of decks:
`winrate` is how often `deck` beat `opponent`, over `games` games, with a Wilson
95% interval. A cell with 6 games and a 100% winrate is noise - the `reliable`
flag (`games >= MIN_GAMES`) and the interval width are there to keep you honest.

**Validity caveats worth knowing before you trust a number**

- *Format drift.* "STANDARD" names a different card pool after every set release.
  `START_DATE` exists to keep the sample inside one metagame; widening it mixes
  incompatible formats into the same cell.
- *Skill is confounded with deck.* A deck piloted mostly by strong players will
  look strong. This measures deck-plus-pilot, not deck.
- *Top cut oversamples winners* and is often a different match format from Swiss.
  `SWISS_ONLY` and the `phaseType` / `phaseMode` columns let you split them.
- *Archetype labels are Limitless's*, and `other` is a bucket rather than a deck,
  so it is dropped by default via `EXCLUDE_DECKS`.

In [ ]:
import json
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# --- what to pull -------------------------------------------------------------
BASE_URL = "https://play.limitlesstcg.com/api/tournaments"
GAME = "PTCG"
FORMAT = "STANDARD"        # a moving target - pin the metagame with START_DATE
MAX_PAGES = 99999              # 50 tournaments per page, newest first
START_DATE = "2025-01-01"  # set-release boundary; mixing formats invalidates the matrix
END_DATE = None            # None = up to today
MIN_PLAYERS = 8           # small casual events are mostly noise

# --- politeness / robustness --------------------------------------------------
REQUEST_TIMEOUT = 30       # seconds to wait on a single HTTP request
SLEEP_BETWEEN = 2       # seconds between live requests - raise if 429s persist
MAX_RETRIES = 5            # attempts per URL before giving up on a 429

# --- analysis knobs -----------------------------------------------------------
MIN_GAMES = 10             # below this a matchup cell is not worth reading
SWISS_ONLY = False         # True drops top cut, which oversamples winning decks
EXCLUDE_DECKS = {"other", ""}  # 'other' is a bucket, not an archetype

CACHE_DIR = Path("cache")
EXPORT_DIR = Path("exports")


class RateLimited(RuntimeError):
    """The API kept replying 429 after MAX_RETRIES attempts."""


def _build_session():
    """A pooled session that retries transport errors and 5xx on its own.

    429 is deliberately NOT in status_forcelist - fetch_json handles it so the
    wait is visible and Retry-After is honoured.
    """
    session = requests.Session()
    retry = Retry(
        total=3,
        status_forcelist=(500, 502, 503, 504),
        allowed_methods=frozenset(["GET"]),
        backoff_factor=1.0,
        raise_on_status=False,
    )
    session.mount("https://", HTTPAdapter(max_retries=retry, pool_maxsize=4))
    session.headers.update({"User-Agent": "limitless-tcg-datascience (research)"})
    return session


SESSION = _build_session()


def _retry_after(response, attempt):
    """Seconds to wait before retrying: honour Retry-After, else exponential backoff."""
    header = response.headers.get("Retry-After")
    if header:
        try:
            return float(header)
        except ValueError:
            pass
    return SLEEP_BETWEEN * (2 ** attempt)


def _cache_path(key):
    return CACHE_DIR / f"{key}.json"


def fetch_json(url, params=None, cache_key=None):
    """GET url and return the decoded body, backing off and retrying on 429.

    With cache_key the response is stored under cache/ and reused forever, which
    is safe only for tournaments that have finished - see cache_key_for().
    Raises RateLimited if the limit outlasts MAX_RETRIES; every other HTTP or
    transport problem propagates as the usual requests exception.
    """
    if cache_key is not None and _cache_path(cache_key).exists():
        return json.loads(_cache_path(cache_key).read_text(encoding="utf-8"))

    for attempt in range(MAX_RETRIES):
        response = SESSION.get(url, params=params, timeout=REQUEST_TIMEOUT)

        if response.status_code == 429:
            wait = _retry_after(response, attempt)
            print(f"  429 - waiting {wait:.1f}s (attempt {attempt + 1}/{MAX_RETRIES})")
            time.sleep(wait)
            continue

        response.raise_for_status()
        payload = response.json()

        if cache_key is not None:
            path = _cache_path(cache_key)
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_text(json.dumps(payload), encoding="utf-8")
        return payload

    raise RateLimited(f"Still rate limited after {MAX_RETRIES} attempts: {url}")


def cache_key_for(endpoint, tournament_id, tournament_date):
    """Cache key for a finished tournament, None for one that may still be running.

    Caching a tournament mid-run would freeze a half-finished bracket on disk.
    """
    if tournament_date >= datetime.now().strftime("%Y-%m-%d"):
        return None
    return f"{endpoint}/{tournament_id}"


def fetch_tournament_endpoint(tournaments, endpoint):
    """Fetch /<endpoint> for every row of `tournaments` (needs `id` and `date`).

    Returns {tournament_id: payload}, [failed ids]. One tournament failing is
    recorded and skipped, but callers must surface the count: a silently
    truncated sample reweights the matchup matrix without looking wrong.

    A surviving 429 is different - it means the API is refusing us, not that this
    tournament is broken, so the sweep stops instead of hammering on. Everything
    already fetched is on disk, so re-running later resumes for free.
    """
    payloads, failures = {}, []
    total = len(tournaments)
    print(f"Fetching {endpoint} for {total} tournaments")

    for i, row in enumerate(tournaments.itertuples(index=False), start=1):
        key = cache_key_for(endpoint, row.id, row.date)
        from_cache = key is not None and _cache_path(key).exists()

        try:
            payloads[row.id] = fetch_json(f"{BASE_URL}/{row.id}/{endpoint}", cache_key=key)
        except RateLimited:
            print(f"  [{i}/{total}] still rate limited - stopping the sweep. "
                  f"{len(payloads)} responses are cached; wait a while and re-run to resume.")
            raise
        except requests.exceptions.RequestException as e:
            print(f"  [{i}/{total}] failed {row.id}: {e}")
            failures.append(row.id)
            continue

        if not from_cache:
            time.sleep(SLEEP_BETWEEN)
        if i % 25 == 0 or i == total:
            print(f"  [{i}/{total}] done")

    if failures:
        print(f"  WARNING: {len(failures)} of {total} tournaments failed and are missing")
    return payloads, failures


def norm_handle(series):
    """Normalise player handles once, so cross-endpoint joins cannot miss on case."""
    return series.astype("string").str.strip().str.lower()


def _flatten_nested(dataframe):
    """Serialize nested values (lists/dicts) so the Excel writer accepts them."""
    out = dataframe.copy()
    nested = (list, dict, set, tuple)
    for col in out.columns:
        if out[col].map(lambda v: isinstance(v, nested)).any():
            out[col] = out[col].map(
                lambda v: json.dumps(v, default=str) if isinstance(v, nested) else v
            )
    return out


def export_df(dataframe, name, export_dir=EXPORT_DIR, allow_empty=False):
    """Write the DataFrame to <name>_<YYYY-MM-DD>_<HHMMSS>.csv and .xlsx.

    Refuses an empty frame by default: a fetch that died on its first request
    would otherwise write a 0-row file indistinguishable from a good run.
    """
    if dataframe.empty and not allow_empty:
        raise ValueError(
            f"Refusing to export empty '{name}' - the fetch that built it produced no rows"
        )

    export_dir.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now().strftime("%Y-%m-%d_%H%M%S")
    csv_path = export_dir / f"{name}_{stamp}.csv"
    xlsx_path = export_dir / f"{name}_{stamp}.xlsx"

    flat = _flatten_nested(dataframe)
    flat.to_csv(csv_path, index=False)
    flat.to_excel(xlsx_path, index=False)

    print(f"Exported {name} ({len(dataframe)} rows) -> {csv_path} | {xlsx_path}")
    return csv_path, xlsx_path


print(f"Config: {FORMAT} {GAME}, <= {MAX_PAGES} pages, from {START_DATE}, "
      f"{MIN_PLAYERS}+ players, cache at {CACHE_DIR.resolve()}")

Config: STANDARD PTCG, <= 5 pages, from 2026-05-01, 16+ players, cache at C:\Users\rntbz\Documents\GitHub\limitless_tcg_datascience\cache


## 1. Tournament list

The listing is never cached - new tournaments appear at the front of page 1 all
the time. Paging stops early once a page is entirely older than `START_DATE`.

In [2]:
raw_tournaments = []
pages_read = 0

for page in range(1, MAX_PAGES + 1):
    print(f"Fetching tournament list page {page}")
    payload = fetch_json(BASE_URL, params={"page": page, "game": GAME, "format": FORMAT, "limit": 100000})

    if not payload:
        print("  empty page - stopping")
        break

    raw_tournaments.extend(payload)
    pages_read += 1

    # Newest first, so once a whole page predates the window there is nothing left to want
    if START_DATE and max(t["date"] for t in payload) < START_DATE:
        print("  page is entirely older than START_DATE - stopping")
        break

    time.sleep(SLEEP_BETWEEN)

all_tournaments_df = pd.DataFrame(raw_tournaments)
print(f"{len(all_tournaments_df)} tournaments from {pages_read} page(s)")

Fetching tournament list page 1
Fetching tournament list page 2
  empty page - stopping
12326 tournaments from 1 page(s)


In [3]:
all_tournaments_df = all_tournaments_df.drop(columns=["game"], errors="ignore")
all_tournaments_df["date"] = pd.to_datetime(all_tournaments_df["date"], utc=True).dt.strftime("%Y-%m-%d")

# The list shifts under us while we page through it, so the same event can arrive twice
before = len(all_tournaments_df)
all_tournaments_df = all_tournaments_df.drop_duplicates(subset="id", keep="first")
if before != len(all_tournaments_df):
    print(f"Dropped {before - len(all_tournaments_df)} duplicate tournaments from overlapping pages")

keep = all_tournaments_df["players"] >= MIN_PLAYERS
print(f"{(~keep).sum()} tournaments under {MIN_PLAYERS} players dropped")

if START_DATE:
    in_window = all_tournaments_df["date"] >= START_DATE
    print(f"{(~in_window).sum()} tournaments before {START_DATE} dropped")
    keep &= in_window
if END_DATE:
    in_window = all_tournaments_df["date"] <= END_DATE
    print(f"{(~in_window).sum()} tournaments after {END_DATE} dropped")
    keep &= in_window

tournaments_df = all_tournaments_df[keep].sort_values("date").reset_index(drop=True)

if tournaments_df.empty:
    raise RuntimeError("No tournaments survived the filters - loosen START_DATE or MIN_PLAYERS")

print(f"\n{len(tournaments_df)} tournaments kept, "
      f"{tournaments_df['date'].min()} to {tournaments_df['date'].max()}, "
      f"{int(tournaments_df['players'].sum())} player entries")

export_df(tournaments_df, "tournaments")
tournaments_df.head()

3691 tournaments under 16 players dropped
11338 tournaments before 2026-05-01 dropped

661 tournaments kept, 2026-05-01 to 2026-07-24, 49078 player entries
Exported tournaments (661 rows) -> exports\tournaments_2026-07-24_194443.csv | exports\tournaments_2026-07-24_194443.xlsx


,name,date,format,id,players,organizerId
0,Le Tournois Du DresseurPokeCharbie #109,2026-05-01,STANDARD,69eac70e78b359b789d6a8ff,24,1689
1,🎮 Free Night – R$30 + Top 10 Codes,2026-05-01,STANDARD,69f4ecf3e23aab068aad75a2,43,2563
2,Peladão da Baixada #169 • Munikis Zero,2026-05-01,STANDARD,69c42b64d478313a15a327da,18,72
3,KickOff - Mesa1 Podcast,2026-05-01,STANDARD,69ebee16519a1682506d65d9,30,2664
4,Vault Weekly #13🏆|L.A REGIONAL PREP|50+ CODES,2026-05-01,STANDARD,69f23ee34f71d664893e72fc,42,2226


## 2. Tournament details and phases

`details` is what tells us a Swiss round apart from top cut, and BO1 apart from
BO3 - both of which change what a winrate means.

In [ ]:
details_payloads, details_failures = fetch_tournament_endpoint(tournaments_df, "details")

details_df = pd.DataFrame(list(details_payloads.values()))
details_df = details_df.drop(columns=["game"], errors="ignore")
details_df["date"] = pd.to_datetime(details_df["date"], utc=True).dt.strftime("%Y-%m-%d")
details_df.head()

Fetching details for 661 tournaments


In [ ]:
# One row per tournament phase, with the phase fields as columns
phases_df = details_df.explode("phases", ignore_index=True)

phase_cols = pd.json_normalize(
    phases_df["phases"].map(lambda v: v if isinstance(v, dict) else {})
).reindex(columns=["phase", "type", "rounds", "mode"])

phases_df = phases_df.drop(columns=["phases"]).join(phase_cols)
phases_df["phase"] = pd.to_numeric(phases_df["phase"], errors="coerce").astype("Int64")

print(phases_df.groupby(["type", "mode"], dropna=False).size().to_string())

export_df(phases_df, "phases")
phases_df.head()

## 3. Pairings - who beat whom

One row per match. Handles are normalised here, once, so the join against
standings later cannot miss on stray case or whitespace.

In [ ]:
pairings_payloads, pairings_failures = fetch_tournament_endpoint(tournaments_df, "pairings")

pairings = []
for tournament_id, payload in pairings_payloads.items():
    # tournamentId last so our value wins if the API ever sends one of its own
    for match in payload:
        pairings.append({**match, "tournamentId": tournament_id})

pairings_df = pd.DataFrame(pairings).reindex(
    columns=["tournamentId", "phase", "round", "table", "player1", "player2", "winner"]
)

for col in ("player1", "player2", "winner"):
    pairings_df[col] = norm_handle(pairings_df[col])
pairings_df["phase"] = pd.to_numeric(pairings_df["phase"], errors="coerce").astype("Int64")

# player2 is absent on byes; a bye is not a game and must never reach the matrix
pairings_df["isBye"] = pairings_df["player2"].isna()

before = len(pairings_df)
pairings_df = pairings_df.drop_duplicates(
    subset=["tournamentId", "phase", "round", "table", "player1", "player2"]
)
if before != len(pairings_df):
    print(f"Dropped {before - len(pairings_df)} duplicate match rows")

print(f"{len(pairings_df)} matches across {pairings_df['tournamentId'].nunique()} tournaments "
      f"({int(pairings_df['isBye'].sum())} byes)")

export_df(pairings_df, "pairings")
pairings_df.head()

## 4. Standings - what each player was on

This is the only source of deck identity, so any tournament missing from here
contributes nothing to the matrix no matter how many pairings it has.

In [ ]:
standings_payloads, standings_failures = fetch_tournament_endpoint(tournaments_df, "standings")

standings = []
for tournament_id, payload in standings_payloads.items():
    for position, entry in enumerate(payload, start=1):
        deck = entry.get("deck") or {}
        record = entry.get("record") or {}

        # The API leaves `placing` out of the entries we have seen; standings come
        # back in finishing order, so fall back to the position in the response.
        placing = entry.get("placing")
        if placing is None:
            placing = position

        standings.append({
            "tournamentId": tournament_id,
            "placing": placing,
            "player": entry.get("player"),
            "name": entry.get("name"),
            "country": entry.get("country"),
            "deckId": deck.get("id"),
            "deckName": deck.get("name"),
            "deckIcons": deck.get("icons"),
            "wins": record.get("wins"),
            "losses": record.get("losses"),
            "ties": record.get("ties"),
            "drop": entry.get("drop"),
            "decklist": entry.get("decklist"),
        })

standings_df = pd.DataFrame(standings, columns=[
    "tournamentId", "placing", "player", "name", "country", "deckId", "deckName",
    "deckIcons", "wins", "losses", "ties", "drop", "decklist",
])

standings_df["player"] = norm_handle(standings_df["player"])
standings_df["deckId"] = standings_df["deckId"].astype("string").str.strip().str.lower()

before = len(standings_df)
standings_df = standings_df.drop_duplicates(subset=["tournamentId", "player"], keep="first")
if before != len(standings_df):
    print(f"Dropped {before - len(standings_df)} duplicate standings rows")

named = ~standings_df["deckId"].isin(EXCLUDE_DECKS) & standings_df["deckId"].notna()
print(f"{len(standings_df)} entries, {standings_df['deckId'].nunique()} distinct decks, "
      f"{(~named).sum()} entries with no usable archetype")

export_df(standings_df, "standings")
standings_df.head()

In [ ]:
# One row per card in every decklist, tagged with its category (pokemon/trainer/energy)
cards = []

for row in standings_df.itertuples(index=False):
    decklist = row.decklist if isinstance(row.decklist, dict) else {}
    for category, entries in decklist.items():
        for card in entries or []:
            cards.append({
                "tournamentId": row.tournamentId,
                "player": row.player,
                "placing": row.placing,
                "deckId": row.deckId,
                "category": category,
                "count": card.get("count"),
                "name": card.get("name"),
                "set": card.get("set"),
                "number": card.get("number"),
            })

decklist_cards_df = pd.DataFrame(cards, columns=[
    "tournamentId", "player", "placing", "deckId", "category", "count", "name", "set", "number",
])

if decklist_cards_df.empty:
    print("No decklists published for this sample - skipping card export")
else:
    # A legal deck is exactly 60 cards; anything else means a partial or odd list
    sizes = decklist_cards_df.groupby(["tournamentId", "player"])["count"].sum()
    odd = int((sizes != 60).sum())
    print(f"{len(decklist_cards_df)} card rows from {len(sizes)} decklists; "
          f"{odd} are not exactly 60 cards")
    export_df(decklist_cards_df, "decklist_cards")

decklist_cards_df.head()

## 5. One row per played game

Each match is turned into two rows, one from each deck's point of view, scored
`1` win / `0` loss / `0.5` tie. Aggregating that gives an antisymmetric matrix
for free - `winrate(A,B)` and `winrate(B,A)` cannot drift apart, because they are
computed from the same rows.

Dropped here: byes (not a game), matches with no result, players missing from
standings (no deck), and mirrors (structurally 50%, and they would double-count
into a single cell).

In [ ]:
# Byes are not games; tournaments with no standings have no decks to attribute
covered = set(standings_df["tournamentId"])
matches = pairings_df[~pairings_df["isBye"] & pairings_df["tournamentId"].isin(covered)].copy()

paired = set(pairings_df["tournamentId"])
print(f"{len(covered & paired)} tournaments have both pairings and standings; "
      f"{len(paired - covered)} pairing-only tournaments dropped")

# On a decided match `winner` holds a player handle; the API uses sentinels
# instead - '0' for a tie, '-1' for no result (unfinished, double loss, DQ).
is_p1 = matches["winner"].eq(matches["player1"])
is_p2 = matches["winner"].eq(matches["player2"])
is_tie = matches["winner"].eq("0")
matches["result"] = np.select([is_p1, is_p2, is_tie], ["p1", "p2", "tie"], default="none")

print(matches["result"].value_counts().to_string())
matches = matches[matches["result"] != "none"].copy()

# Deck per (tournament, player), attached to each side in turn
deck_lookup = standings_df[["tournamentId", "player", "deckId"]].drop_duplicates(
    subset=["tournamentId", "player"]
)
for side, deck_col in (("player1", "deck1"), ("player2", "deck2")):
    matches = matches.merge(
        deck_lookup.rename(columns={"player": side, "deckId": deck_col}),
        on=["tournamentId", side],
        how="left",
    )

no_deck = matches[["deck1", "deck2"]].isna().any(axis=1)
if no_deck.any():
    print(f"{int(no_deck.sum())} matches dropped: a player has no known deck "
          f"(missing from standings, or no archetype recorded)")
matches = matches[~no_deck]

# Phase type/mode: top cut is not Swiss, and BO3 is not BO1
phase_lookup = phases_df[["id", "phase", "type", "mode"]].rename(
    columns={"id": "tournamentId", "type": "phaseType", "mode": "phaseMode"}
).drop_duplicates(subset=["tournamentId", "phase"])
matches = matches.merge(phase_lookup, on=["tournamentId", "phase"], how="left")

unmatched = int(matches["phaseType"].isna().sum())
if unmatched:
    print(f"NOTE: {unmatched} matches could not be tied to a phase record")

if SWISS_ONLY:
    before = len(matches)
    matches = matches[matches["phaseType"].eq("SWISS")]
    print(f"SWISS_ONLY: dropped {before - len(matches)} top-cut matches")

print(f"\n{len(matches)} usable games")
matches.head()

In [ ]:
# Winner-first view of every decided game - one row per match, both decks named
decided = matches[matches["result"] != "tie"].copy()
winner_is_p1 = decided["result"].eq("p1")

master_df = pd.DataFrame({
    "tournament_id": decided["tournamentId"],
    "phase_number": decided["phase"],
    "phase_type": decided["phaseType"],
    "phase_mode": decided["phaseMode"],
    "player1": decided["winner"],                                              # WINNER
    "player1_deckid": decided["deck1"].where(winner_is_p1, decided["deck2"]),
    "player2": decided["player2"].where(winner_is_p1, decided["player1"]),     # LOSER
    "player2_deckid": decided["deck2"].where(winner_is_p1, decided["deck1"]),
}).reset_index(drop=True)

assert (master_df["player1"] != master_df["player2"]).all(), "a player cannot beat themselves"

export_df(master_df, "master")
master_df.head()

In [ ]:
# Two rows per match, one per side. score: 1 win / 0.5 tie / 0 loss.
score_p1 = np.select([matches["result"].eq("p1"), matches["result"].eq("tie")], [1.0, 0.5], default=0.0)

side_a = pd.DataFrame({
    "tournamentId": matches["tournamentId"], "phase": matches["phase"],
    "phaseType": matches["phaseType"], "phaseMode": matches["phaseMode"],
    "deck": matches["deck1"], "opponent": matches["deck2"], "score": score_p1,
})
side_b = pd.DataFrame({
    "tournamentId": matches["tournamentId"], "phase": matches["phase"],
    "phaseType": matches["phaseType"], "phaseMode": matches["phaseMode"],
    "deck": matches["deck2"], "opponent": matches["deck1"], "score": 1.0 - score_p1,
})

games_df = pd.concat([side_a, side_b], ignore_index=True)

# 'other' is a bucket of unlike decks - a winrate against it means nothing
bucketed = games_df["deck"].isin(EXCLUDE_DECKS) | games_df["opponent"].isin(EXCLUDE_DECKS)
print(f"{int(bucketed.sum())} game-sides dropped for an excluded/unlabelled archetype")
games_df = games_df[~bucketed]

# Mirrors are 50% by construction and would land both sides in the same cell
mirrors = games_df["deck"].eq(games_df["opponent"])
mirror_games = int(mirrors.sum()) // 2
print(f"{mirror_games} mirror matches excluded from the matrix")
games_df = games_df[~mirrors].reset_index(drop=True)

print(f"\n{len(games_df)} game-sides = {len(games_df) // 2} matches over "
      f"{games_df['deck'].nunique()} decks")
export_df(games_df, "games")
games_df.head()

## 6. The matchup matrix

`winrate` is `(wins + 0.5 * ties) / games`. The Wilson interval is the part to
actually look at: 4-1 and 40-10 are both "80%", and only one of them means
anything.

In [ ]:
def wilson_interval(wins, games, z=1.96):
    """Wilson score interval for a winrate - honest at the small samples this data is full of.

    Ties count as half a win, so `wins` can be fractional and the interval is then
    an approximation rather than an exact binomial bound.
    """
    games = np.asarray(games, dtype=float)
    wins = np.asarray(wins, dtype=float)
    with np.errstate(invalid="ignore", divide="ignore"):
        p = np.where(games > 0, wins / games, np.nan)
        denom = 1 + z ** 2 / games
        centre = (p + z ** 2 / (2 * games)) / denom
        margin = z * np.sqrt((p * (1 - p) + z ** 2 / (4 * games)) / games) / denom
    return centre - margin, centre + margin


matchup_df = (
    games_df.groupby(["deck", "opponent"], as_index=False)
    .agg(games=("score", "size"), wins=("score", "sum"))
)
matchup_df["losses"] = matchup_df["games"] - matchup_df["wins"]
matchup_df["winrate"] = matchup_df["wins"] / matchup_df["games"]
matchup_df["ci_low"], matchup_df["ci_high"] = wilson_interval(matchup_df["wins"], matchup_df["games"])
matchup_df["ci_width"] = matchup_df["ci_high"] - matchup_df["ci_low"]
matchup_df["reliable"] = matchup_df["games"] >= MIN_GAMES

matchup_df = matchup_df.sort_values(["games", "winrate"], ascending=False).reset_index(drop=True)

print(f"{len(matchup_df)} ordered deck pairs, "
      f"{int(matchup_df['reliable'].sum())} with at least {MIN_GAMES} games")

export_df(matchup_df, "matchups")
matchup_df.head(15)

In [ ]:
# Structural checks - if any of these fire, the numbers above are wrong, not just noisy
reverse = matchup_df.merge(
    matchup_df, left_on=["deck", "opponent"], right_on=["opponent", "deck"], suffixes=("", "_rev")
)
assert len(reverse) == len(matchup_df), "every pair must have its mirror image"
assert (reverse["games"] == reverse["games_rev"]).all(), "A-vs-B and B-vs-A must see the same games"
assert np.allclose(reverse["winrate"] + reverse["winrate_rev"], 1.0), "winrates must be antisymmetric"
assert np.isclose(matchup_df["games"].sum(), 2 * (len(games_df) // 2)), "every game counted exactly twice"
assert matchup_df["winrate"].between(0, 1).all()
assert (matchup_df["deck"] != matchup_df["opponent"]).all(), "mirrors should have been removed"
assert not matchup_df.duplicated(subset=["deck", "opponent"]).any()

print("All structural checks passed:")
print(f"  {len(games_df) // 2} matches -> {matchup_df['games'].sum()} counted game-sides")
print(f"  {matchup_df['deck'].nunique()} decks, {len(matchup_df)} ordered pairs")
print(f"  median games per pair: {matchup_df['games'].median():.0f}, "
      f"max: {matchup_df['games'].max()}")

## 7. Deck summary and the readable matrix

`deck_summary` is the overall picture per deck: how much of the field it was
(meta share, from standings entries) and how it did overall (from the games
table, mirrors excluded).

The pivot at the end is the human-readable matrix, limited to the most-played
decks and blanked where the sample is too thin to mean anything.

In [ ]:
deck_summary = (
    games_df.groupby("deck", as_index=False)
    .agg(games=("score", "size"), wins=("score", "sum"))
)
deck_summary["winrate"] = deck_summary["wins"] / deck_summary["games"]
deck_summary["ci_low"], deck_summary["ci_high"] = wilson_interval(
    deck_summary["wins"], deck_summary["games"]
)

# Meta share from standings entries, over the tournaments that actually made the matrix
used = set(games_df["tournamentId"])
entries = standings_df[
    standings_df["tournamentId"].isin(used) & ~standings_df["deckId"].isin(EXCLUDE_DECKS)
]
share = entries["deckId"].value_counts(normalize=True).rename("meta_share")
deck_summary = deck_summary.merge(share, left_on="deck", right_index=True, how="left")

deck_summary = deck_summary.sort_values("games", ascending=False).reset_index(drop=True)
export_df(deck_summary, "deck_summary")
deck_summary.head(15)

In [ ]:
# Readable matrix: rows beat columns this often. Blank = fewer than MIN_GAMES games.
TOP_N = 12
top_decks = deck_summary.head(TOP_N)["deck"].tolist()

view = matchup_df[
    matchup_df["deck"].isin(top_decks)
    & matchup_df["opponent"].isin(top_decks)
    & matchup_df["reliable"]
]

matrix = (
    view.pivot(index="deck", columns="opponent", values="winrate")
    .reindex(index=top_decks, columns=top_decks)
)

thin = matrix.isna().sum().sum() - len(top_decks)  # the diagonal is always blank
print(f"Top {len(top_decks)} decks by games played; {thin} pairs blank for under {MIN_GAMES} games\n")

matrix.style.format("{:.0%}", na_rep="-").background_gradient(
    cmap="RdYlGn", vmin=0.3, vmax=0.7, axis=None
).set_caption(f"Row deck's winrate vs column deck ({MIN_GAMES}+ games)")

## 8. The visual matrix

Writes a self-contained `matchup_matrix.html` next to this notebook - open it
straight from disk, no server needed. Re-run this cell after any fetch and the
page reflects the new sample.

In [ ]:
from viz import write_matchup_page

write_matchup_page(matchup_df, deck_summary, standings_df, tournaments_df)